The contents of this notebook were created with assistance from Claude generative AI.

# Label the full corpus with the 5-fold-CV-tuned ModernBERT (Project Data - 15)

Loads `saved_models/tuned/modernbert` (this folder's tuned ModernBERT @512) and predicts stance
`{anti, neutral, pro, off-topic}` for every **scoreable** item (`hp ∪ matched_tier1 ∪ v5-relevant`,
~427k). Inference settings (max_len, truncation side, labels) come from the model's
`inference_config.json` via `m5_common.load_proba`. CONTEXT/TARGET is rebuilt exactly as in training.
Output: **`stance_predictions_modernbert.parquet`** in this P15 folder.

In [4]:
import json
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")          # GPU1 (GPU0 drives the display)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import sys
from pathlib import Path
import numpy as np, pandas as pd

P15 = Path.cwd()
sys.path.insert(0, str(P15))
import m5_common as m5                       # canonical text build + model load/inference

STAGE, MODEL = "tuned", "ModernBERT"
MODEL_PATH = m5.model_path(STAGE, MODEL)      # P15/saved_models/tuned/modernbert
OUT = Path("../data") / "stance_predictions_modernbert_tuned.parquet"
DATA = m5.DATA                                # P11 reduced corpus (comments_reduced / posts_reduced)
assert MODEL_PATH.exists(), f"tuned ModernBERT not found at {MODEL_PATH}"

# truncation limits + junk set — identical to build_relabel_5k_v2.py (matches training text)
TITLE_TRUNC, SELF_TRUNC, ANC_TRUNC, BODY_TRUNC, MAX_ANC = 4000, 8000, 2000, 8000, 8
JUNK = {"", "[deleted]", "[removed]", "[ Removed by Reddit ]", "None", "nan"}
print("model:", MODEL_PATH)
print("inference_config:", json.loads((MODEL_PATH / "inference_config.json").read_text()))
print("out:", OUT)

model: saved_models\tuned\modernbert
inference_config: {'max_len': 512, 'truncation_side': 'left', 'labels': ['anti', 'neutral', 'pro', 'off-topic'], 'hyperparams': {'lr': 4.991029159030355e-05, 'weight_decay': 0.01622890417738726, 'warmup_ratio': 0.038038895602179296, 'epochs': 4, 'class_weighted': True}}
out: stance_predictions_modernbert.parquet


## 1. Load the reduced corpus (scoreable items + their retained ancestors)

In [5]:
C = pd.read_parquet(DATA / "comments_reduced.parquet",
        columns=["id", "parent_id", "link_id", "body", "author", "is_submitter",
                 "subreddit", "created_utc", "role"])
P = pd.read_parquet(DATA / "posts_reduced.parquet",
        columns=["name", "title", "selftext", "author", "subreddit", "created_utc", "role"])
tcm   = {r["id"]:   r for r in C.to_dict("records")}
posts = {r["name"]: r for r in P.to_dict("records")}
print(f"comments={len(C):,}  posts={len(P):,}  scoreable: "
      f"{(C.role=='scoreable').sum():,} comments + {(P.role=='scoreable').sum():,} posts")

comments=504,553  posts=25,505  scoreable: 422,961 comments + 4,075 posts


## 2. Rebuild CONTEXT/TARGET exactly as in training (build_relabel_5k_v2.py)

In [6]:
def trunc(s, n):
    s = str(s or "").strip()
    return s if len(s) <= n else s[:n] + " …[trunc]"

def anon(post_author):
    alias, letters = {}, iter("ABCDEFGHJKLMNPQRSTUVWXYZ")
    def al(a, is_sub=False):
        a = str(a) if a else "[deleted]"
        if a in ("[deleted]", "None", "nan", ""): return "[deleted]"
        if post_author is not None and a == str(post_author): return "OP"
        if is_sub: return "OP"
        if a not in alias: alias[a] = next(letters, "Z")
        return alias[a]
    return al

def chain(cid):
    anc, cur = [], cid
    while True:
        row = tcm.get(cur); pid = row["parent_id"] if row else None
        if not isinstance(pid, str) or not pid.startswith("t1_"): break
        parent = tcm.get(pid[3:])
        if parent is None: break
        anc.append(parent); cur = parent["id"]
    return list(reversed(anc))[-MAX_ANC:]

def comment_text(cid, link_id):
    tgt = tcm.get(cid); p = posts.get(link_id) or {}
    al = anon(p.get("author")); ctx = []
    if p.get("title"):
        ctx.append(f"[POST · OP] {trunc(p.get('title'), TITLE_TRUNC)}")
        if str(p.get("selftext")).strip() not in JUNK:
            ctx.append(f"    {trunc(p.get('selftext'), SELF_TRUNC)}")
    for a in chain(cid):
        ctx.append(f"[{al(a['author'], bool(a.get('is_submitter')))}] {trunc(a['body'], ANC_TRUNC)}")
    tag = al(tgt["author"], bool(tgt.get("is_submitter"))) if tgt else "?"
    target = f"[{tag}] " + (trunc(tgt["body"], BODY_TRUNC) if tgt else "[unavailable]")
    return "\n".join(ctx), target

def post_text(name):
    p = posts.get(name) or {}
    body = "" if str(p.get("selftext")).strip() in JUNK else trunc(p.get("selftext"), SELF_TRUNC)
    return "(top-level post — no parent context)", "[OP] " + f"{trunc(p.get('title'), TITLE_TRUNC)}\n\n{body}".strip()

rows = []
for r in C[C.role == "scoreable"].itertuples():
    ctx, tgt = comment_text(r.id, r.link_id)
    rows.append((r.id, "comment", r.subreddit, r.created_utc, r.link_id, ctx, tgt))
for r in P[P.role == "scoreable"].itertuples():
    ctx, tgt = post_text(r.name)
    bare = r.name[3:] if isinstance(r.name, str) and r.name.startswith("t3_") else r.name
    rows.append((bare, "post", r.subreddit, r.created_utc, r.name, ctx, tgt))
items = pd.DataFrame(rows, columns=["id", "kind", "subreddit", "created_utc", "link_id", "context", "target"])
items = m5._build_text(items)               # adds the "[CONTEXT]\n…\n\n[TARGET]\n…" column, same as training
print("items to label:", len(items))

items to label: 427036


## 3. Predict with the tuned ModernBERT and save to parquet

In [7]:
# m5.load_proba loads saved_models/tuned/modernbert, reads its inference_config (max_len 512,
# left-truncation), and returns (n, 4) probabilities in m5.LABELS order.
proba = m5.load_proba(STAGE, MODEL, items)
items["stance_pred"] = [m5.LABELS[k] for k in proba.argmax(1)]
for i, l in enumerate(m5.LABELS):
    items[f"p_{l.replace('-', '_')}"] = proba[:, i]

out_cols = ["id", "kind", "subreddit", "created_utc", "link_id", "stance_pred"] + \
           [f"p_{l.replace('-', '_')}" for l in m5.LABELS]
items[out_cols].to_parquet(OUT, index=False)
print("wrote", OUT, "  rows:", len(items))
print(items["stance_pred"].value_counts())

<home>\.venvs\mads-m2-classifiers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 6000.19it/s]


wrote stance_predictions_modernbert.parquet   rows: 427036
stance_pred
off-topic    179418
pro          150546
anti          72400
neutral       24672
Name: count, dtype: int64
